In [ ]:
from sentence_transformers import SentenceTransformer
from pathlib import Path
from pypdf import PdfReader
import chromadb
from dotenv import load_dotenv
from groq import Groq
import os

## Load Files

In [42]:
pdf_files = list(Path("data").glob("*.pdf"))
documents = []

for pdf_file in pdf_files:
    reader = PdfReader(pdf_file)
    text = ""

    for page in reader.pages:
        text += page.extract_text() + "\n"

    documents.append({
        "source": pdf_file.name,
        "text": text
    })

documents

[{'source': 'rise-of-llm.pdf',
  'text': 'R&D www.managementsolutions.com\nThe rise of Large Language Models: \nfrom fundamentals to application \nAuge LLM-Eng- Vdef_Maquetación 1  30/05/2024  23:48  Página 1\nDesign and Layout \nMarketing and Communication Department \nManagement Solutions \nPhotographs  \nPhotographic archive of Management Solutions \nAdobeStock \nMidjourney \n \n \n© Management Solutions 2024 \nAll rights reserved. Cannot be reproduced, distributed, publicly disclosed, converted, totally or partially, freely or with a charge, in any way or procedure, without the \nexpress written authorization of Management Solutions. The information contained in this publication is merely to be used as a guideline. Management Solutions shall  \nnot be held responsible for the use which could be made of this information by third parties. Nobody is entitled to use this material except by express authorization of \nManagement Solutions.\nAuge LLM-Eng- Vdef_Maquetación 1  30/05/2024  2

## Create Chunks

In [43]:
chunks = []
chunk_size = 200
chunk_overlap = 20

for doc in documents:

    text = doc["text"]
    source = doc["source"]

    start = 0
    chunk_id = 0

    while start < len(text):

        end = start + chunk_size
        chunk_text = text[start:end]

        chunks.append({
            "text": chunk_text,
            "source": source,
            "chunk_id": chunk_id
        })

        chunk_id += 1
        start += chunk_size - chunk_overlap

len(chunks)

1359

## Embed Chunks

In [44]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3217.54it/s]


In [45]:
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(texts)

for chunk, emb in zip(chunks, embeddings):
    chunk["embedding"] = emb

print("Chunks embedded:", len(embeddings))
print("Embedding size:", len(embeddings[0]))

Chunks embedded: 1359
Embedding size: 384


## Save Embeddings in Chroma

In [46]:
client = chromadb.PersistentClient('./chroma_db')

collection = client.get_or_create_collection(
    name="knowledge_base"
)

documents = [
    chunk["text"] for chunk in chunks
]

embeddings = [
    chunk["embedding"].tolist()
    for chunk in chunks
]
        
ids = [
    f"{chunk['source']}_{chunk['chunk_id']}"
    for chunk in chunks
]

metadatas = [
    {
        "source": chunk["source"]
    }
    for chunk in chunks
]

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

## Embed Query

In [56]:
query = 'What is a LLM?'

embeded_query = model.encode(query)

## Perform Search

In [57]:
n_results = 3

results = collection.query(
    query_embeddings=[
        embeded_query
    ],
    n_results=n_results
)


In [58]:
results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

## Extract Context

In [59]:
retrieved_chunks = results['documents'][0]
retrieved_meta = results["metadatas"][0]

context = ""

for chunk, meta in zip(retrieved_chunks, retrieved_meta):

    context += (
        f"Source: {meta['source']}\n"
        f"Text: {chunk}\n\n"
    )

context

'Source: rise-of-llm.pdf\nText: an LLM depends on its size, the diversity of \nits training data, and the sophistication of its algorithms, which \ndirectly affects its ability to be used in practical applications in \nvarious fields. \n\nSource: rise-of-llm.pdf\nText: dscape and its \nfuture prospects. Through detailed analysis, case studies, and \ndiscussion of current trends and challenges, this paper covers \nkey aspects of the context and definition of LLMs, their\n\nSource: rise-of-llm.pdf\nText: \nLLM: definition, context and regulation\n“I was told I would have a positive impact on the world. No one prepared me for the \namount of ridiculous questions I would be asked on a daily basis“. \nAnthro\n\n'

## Integrate LLM

In [60]:
prompt = f"""
    Answer the question using ONLY the provided context. Also name the source document.

    Context:
    {context}

    Question:
    {query}

    Answer:
"""

In [ ]:
load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)


response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

In [62]:
response.choices[0].message.content

'The provided context does not explicitly define what a LLM is. However, based on the given information, it can be inferred that LLM stands for "Large Language Model" as it is mentioned in the context of training data, algorithms, and practical applications.\n\nSource: rise-of-llm.pdf'